In [2]:
!pip install os

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


In [2]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
  Using cached kagglehub-1.0.1-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.22-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached kagglehub-1.0.1-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.22-py3-none-any.whl (212 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\athay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import kagglehub
import os
os.environ['KAGGLE_USERNAME'] = 'athayanaufal'
os.environ['KAGGLE_KEY'] = 'ffd4e2414842d9aa07e4b650613b758d'

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

100%|██████████| 34.1k/34.1k [00:00<00:00, 4.03MB/s]

Extracting files...
Path to competition files: C:\Users\athay\.cache\kagglehub\competitions\titanic


In [5]:
import pandas as pd

# Read the data
train_file = os.path.join(path, 'train.csv')
test_file = os.path.join(path, 'test.csv')

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

# Display first few rows
train_df.head()
test_df.head()


Train data shape: (891, 12)
Test data shape: (418, 11)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [6]:
# Check missing values
print("Missing values in train data:")
print(train_df.isnull().sum())
print("\nMissing values in test data:")
print(test_df.isnull().sum())

# Percentage of missing values
print("\nPercentage of missing values in train data:")
print((train_df.isnull().sum() / len(train_df)) * 100)
print("\nPercentage of missing values in test data:")
print((test_df.isnull().sum() / len(test_df)) * 100)

Missing values in train data:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing values in test data:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Percentage of missing values in train data:
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

Percentage of missing values in test data:
PassengerId     0.000000
Pclass          0.000000
Name            0.00000

In [8]:
# Fill Age with median
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
test_df['Age'].fillna(train_df['Age'].median(), inplace=True)  # use train median for test

# Fill Embarked with mode
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
test_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

# Fill Fare in test with median
test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)

# Drop Cabin
train_df.drop('Cabin', axis=1, inplace=True)
test_df.drop('Cabin', axis=1, inplace=True)

print("Missing values after filling - Train:", train_df.isnull().sum().sum())
print("Test:", test_df.isnull().sum().sum())
print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)


Missing values after filling - Train: 0
Test: 0

Train shape: (891, 11)
Test shape: (418, 10)


C:\Users\athay\AppData\Local\Temp\ipykernel_3116\4273481409.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
C:\Users\athay\AppData\Local\Temp\ipykernel_3116\4273481409.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

def perform_feature_engineering(df):
 
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
    
    cols_to_drop = ['Name', 'Ticket', 'PassengerId']
    df = df.drop(cols_to_drop, axis=1)
    return df

train_df = perform_feature_engineering(train_df)
test_df = perform_feature_engineering(test_df)

le = LabelEncoder()
train_df['Sex'] = le.fit_transform(train_df['Sex'])
test_df['Sex'] = le.transform(test_df['Sex'])

train_df = pd.get_dummies(train_df, columns=['Embarked', 'Title'])
test_df = pd.get_dummies(test_df, columns=['Embarked', 'Title'])

train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)


X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(test_df.drop('Survived', axis=1))

print("Feature Engineering Selesai!")
print(f"Jumlah fitur sekarang: {X_train.shape[1]}")

<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:7: SyntaxWarning: invalid escape sequence '\.'
C:\Users\athay\AppData\Local\Temp\ipykernel_3116\1961941724.py:7: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Feature Engineering Selesai!
Jumlah fitur sekarang: 16


In [19]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. RANDOM FOREST ---
print("Training Random Forest...")
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

rf_clf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf_clf, rf_param_grid, cv=5, scoring='accuracy')
rf_grid.fit(X_train, y_train) # RF bisa pakai data non-scaled, tapi X_train sudah siap

best_rf = rf_grid.best_estimator_
rf_preds = best_rf.predict(X_val)

# --- 2. XGBOOST ---
print("Training XGBoost...")
xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7]
}

xgb_clf = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_grid = GridSearchCV(xgb_clf, xgb_param_grid, cv=5, scoring='accuracy')
xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
xgb_preds = best_xgb.predict(X_val)

# --- 3. EVALUASI HASIL ---
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n--- {name} Performance ---")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))

evaluate_model("Random Forest", y_val, rf_preds)
evaluate_model("XGBoost", y_val, xgb_preds)

Training Random Forest...
Training XGBoost...


C:\Users\athay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:200: UserWarning: [00:29:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\athay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:200: UserWarning: [00:29:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\athay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:200: UserWarning: [00:29:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } 


--- Random Forest Performance ---
Accuracy: 0.8324
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86       105
           1       0.82      0.76      0.79        74

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179


--- XGBoost Performance ---
Accuracy: 0.8156
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.72      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



C:\Users\athay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:200: UserWarning: [00:29:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# --- 1. MEMBANGUN ARSITEKTUR MLP ---
def build_mlp(input_dim):
    model = Sequential([
        # Hidden Layer 1
        Dense(32, activation='relu', input_dim=input_dim),
        Dropout(0.2), # Regularization untuk mencegah overfitting
        
        # Hidden Layer 2
        Dense(16, activation='relu'),
        Dropout(0.2),
        
        # Output Layer (Binary Classification: Survived or Not)
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# --- 2. TRAINING MODEL ---
# Gunakan X_train_scaled yang sudah kita buat sebelumnya
mlp_model = build_mlp(X_train_scaled.shape[1])

# Early Stopping agar training berhenti saat akurasi validasi tidak naik lagi
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training MLP (Deep Learning)...")
history = mlp_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stop],
    verbose=0 # Set ke 1 jika ingin melihat progress tiap epoch
)

# --- 3. EVALUASI MLP ---
mlp_loss, mlp_acc = mlp_model.evaluate(X_val_scaled, y_val, verbose=0)
print(f"\n--- MLP Performance ---")
print(f"Accuracy: {accuracy_score(y_val, mlp_preds):.4f}")
print("Classification Report:")
print(classification_report(y_val, mlp_preds))

# Prediksi untuk laporan (ubah probabilitas ke kelas 0/1)
mlp_preds = (mlp_model.predict(X_val_scaled) > 0.5).astype("int32")

C:\Users\athay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training MLP (Deep Learning)...

--- MLP Performance ---
Accuracy: 0.8212
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.88      0.85       105
           1       0.81      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
